# Few-shot example clips

Transcribes 40 candidate FLEURS clips with the three recognisers, for choosing the five clips used as examples in the few-shot prompts.

- **Pool:** FLEURS `ur_pk` test and validation clips that are not among the 1,200 scored FLEURS clips. The scored clips are the first 1,200 of the seed-42 shuffle of test + validation + train, as in the baseline, so the pool is the rest of that shuffle. Train clips are left out because Omnilingual was trained on them. Only clips whose reference has 3 to 25 words are kept.
- **Order:** the 40 candidates are the first pool clips in shuffled order.

| File | Contents |
|---|---|
| `data/few_shot_candidates/samples_few_shot_candidates.csv` | The 40 transcribed clips |
| `data/few_shot_candidates/<recogniser>_transcripts.csv` | Transcripts of the 40 clips (also the checkpoint) |
| `data/few_shot_candidates/run_info.json` | Machine, package versions, models |

## 1. Setup

In [ ]:
import os, sys, json, glob, time, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

sh("pip install -q jiwer soundfile librosa sentencepiece")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")

### Settings, normaliser and checkpoint helpers

In [ ]:
%%writefile baseline_common.py
import os, re, glob, json, time, shutil, string, unicodedata
import pandas as pd

OUT = "/kaggle/working/output"
DATA = f"{OUT}/data"
AUDIO_DIR = "/tmp/audio_16k"   # rebuilt each session; not kept as output
CANDIDATE_DIR = f"{DATA}/few_shot_candidates"
SAMPLE_FILES = {"candidates": f"{CANDIDATE_DIR}/samples_few_shot_candidates.csv"}
SAMPLE_RATE = 16000
MAX_CLIP_SECONDS = 40          # Omnilingual's limit, applied to all recognisers so they score the same clips
CHECKPOINT_EVERY = 50

RECOGNISERS = {
    "whisper_large_v3": "Whisper-large-v3",
    "omnilingual_llm_1b_v2": "Omnilingual LLM 1B v2",
    "indic_transcribe_core": "Indic Transcribe Core",
}
TRANSCRIPT_COLUMNS = ["clip_id", "transcript", "time_s", "status", "decoding", "word_confidence"]

# clean_urdu v1
NORMALISER_VERSION = "clean_urdu v1"
_DIAC_RE = re.compile("[ؐ-ؚـً-ٰۖ-ۜ۟-۪ۨ-ۭ]")
_PUNCT = set(string.punctuation) | set("۔،؟؛٪।॥“”"
                                       "‘’—–…«»‖‐‒·")
_IZAFAT = {"اے", "ای", "ئے", "ء", "ے", "او"}
_UMAP = {"ي": "ی", "ك": "ک", "ة": "ہ", "ۃ": "ہ",
         "ى": "ی", "إ": "ا", "أ": "ا", "آ": "ا",
         "ٲ": "ا", "ٳ": "ا", "ؤ": "و", "ئ": "ی"}

def clean_urdu(text):
    t = unicodedata.normalize("NFC", str(text))
    t = _DIAC_RE.sub("", t)
    t = "".join(_UMAP.get(ch, ch) for ch in t)
    t = "".join(" " if ch in _PUNCT else ch for ch in t)
    return " ".join(w for w in t.split() if w not in _IZAFAT).strip()


def audio_path(clip_id):
    return f"{AUDIO_DIR}/{clip_id.replace('/', '__')}.wav"


def load_samples(included_only=False):
    clips = pd.concat([pd.read_csv(p, keep_default_na=False) for p in SAMPLE_FILES.values()], ignore_index=True)
    if included_only:
        clips = clips[clips.included.astype(str) == "True"].reset_index(drop=True)
    return clips


def transcript_path(stem):
    return f"{CANDIDATE_DIR}/{stem}_transcripts.csv"


def load_transcripts(stem):
    # Transcripts of a previous run attached as a Kaggle input are copied in first, so the run continues from them.
    path = transcript_path(stem)
    if not os.path.exists(path):
        earlier = glob.glob(f"/kaggle/input/**/data/few_shot_candidates/{stem}_transcripts.csv", recursive=True)
        if earlier:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            shutil.copy(earlier[0], path)
            print(f"Continuing {RECOGNISERS[stem]} from {earlier[0]}")
    if os.path.exists(path):
        return pd.read_csv(path, keep_default_na=False)
    return pd.DataFrame(columns=TRANSCRIPT_COLUMNS)


def append_transcripts(stem, rows):
    path = transcript_path(stem)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pd.DataFrame(rows, columns=TRANSCRIPT_COLUMNS).to_csv(
        path, mode="a", header=not os.path.exists(path), index=False, encoding="utf-8")


def run_recogniser(stem, transcribe_one):
    # transcribe_one(path, duration_s) returns {text, decoding, word_confidence, status};
    # an exception is saved as status 'error: ...' with an empty transcript.
    clips = load_samples(included_only=True)
    done = set(load_transcripts(stem).clip_id)
    todo = clips[~clips.clip_id.isin(done)]
    name = RECOGNISERS[stem]
    print(f"{name}: {len(done)} clips already done, {len(todo)} to go")
    rows, t_start = [], time.perf_counter()
    for k, clip in enumerate(todo.itertuples(), 1):
        t0 = time.perf_counter()
        try:
            r = transcribe_one(audio_path(clip.clip_id), float(clip.duration_s))
            status = r["status"] if r["text"].strip() else "empty_output"
            row = [clip.clip_id, r["text"], 0.0, status, r["decoding"],
                   json.dumps(r["word_confidence"], ensure_ascii=False) if r["word_confidence"] else ""]
        except Exception as e:
            row = [clip.clip_id, "", 0.0, f"error: {type(e).__name__}: {e}"[:300], "", ""]
        row[2] = round(time.perf_counter() - t0, 3)
        rows.append(row)
        if len(rows) == CHECKPOINT_EVERY or k == len(todo):
            append_transcripts(stem, rows)
            rows = []
            rate = (time.perf_counter() - t_start) / k
            print(f"  {name}: {len(done) + k}/{len(clips)} clips saved | {rate:.2f} s per clip", flush=True)
    print(f"{name}: finished")

### Python 3.10 environment for Omnilingual and Whisper
Built once per session.

In [ ]:
PY = "/kaggle/working/py310/bin/python"
if not os.path.exists(PY):
    sh("pip install -q uv && uv python install 3.10 && uv venv --seed --python 3.10 /kaggle/working/py310")
    sh(f"{PY} -m pip install -q --index-url https://download.pytorch.org/whl/cu128 torch==2.8.0 torchaudio==2.8.0")
    sh(f"{PY} -m pip install -q --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu128 fairseq2n==0.6.0 fairseq2==0.6.0")
    sh(f"{PY} -m pip install -q --ignore-requires-python --no-deps omnilingual-asr==0.2.0")
    sh(f'{PY} -m pip install -q polars sentencepiece editdistance retrying "pandas==2.2.3" "numpy>=1.26,<2.3" '
       f'"pyarrow==20.0.0" "fsspec<=2024.9.0" "datasets==3.2.0" "transformers>=4.46,<4.49" jiwer librosa soundfile accelerate')
    sh(f'{PY} -m pip install -q "huggingface_hub>=0.32,<0.33"')  # fairseq2 needs huggingface_hub 0.32.x; install last

sh(f'{PY} -c "from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline; '
   f"import torch, transformers; print('Python 3.10 env OK | torch', torch.__version__, '| transformers', transformers.__version__)\"")

## 2. Candidate clips and audio

FLEURS test, validation and train are joined in that order and shuffled with seed 42, as in the baseline. The first 1,200 are the scored clips and are skipped. From the rest, test and validation clips with 3 to 25 words are kept and the first 40 are saved.
Audio is saved as 16 kHz mono 32-bit float WAV.

In [ ]:
import io
import numpy as np, pandas as pd, soundfile as sf, librosa
from huggingface_hub import list_repo_files, hf_hub_download
from baseline_common import *

SEED, SKIP_FIRST, N_CANDIDATES, MIN_WORDS, MAX_WORDS = 42, 1200, 40, 3, 25
os.makedirs(AUDIO_DIR, exist_ok=True)


def to_16k(audio, sr):
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio if sr == SAMPLE_RATE else librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)


def save_audio(clip_id, audio):
    path = audio_path(clip_id)
    if not os.path.exists(path):
        sf.write(path, audio, SAMPLE_RATE, subtype="FLOAT")
    return round(len(audio) / SAMPLE_RATE, 2)


def hf_download(repo, path, **kw):
    for attempt in range(10):
        try:
            return hf_hub_download(repo, path, repo_type="dataset", **kw)
        except Exception as e:
            if "429" not in str(e):
                raise
            print(f"  rate limited, waiting {30 * (attempt + 1)} s", flush=True)
            time.sleep(30 * (attempt + 1))
    raise RuntimeError(f"still rate limited: {path}")




def fleurs_candidates():
    files = list_repo_files("google/fleurs", repo_type="dataset", revision="refs/convert/parquet")
    parts = []
    for split in ["test", "validation", "train"]:
        paths = sorted(f for f in files if f.startswith(f"ur_pk/{split}/") and f.endswith(".parquet"))
        df = pd.concat([pd.read_parquet(hf_download("google/fleurs", p, revision="refs/convert/parquet"))
                        for p in paths], ignore_index=True)
        df["split"] = split
        parts.append(df)
        print(f"  FLEURS {split}: {len(df)} rows")
    pool = pd.concat(parts, ignore_index=True)
    order = np.random.default_rng(SEED).permutation(len(pool))[SKIP_FIRST:]
    rows = []
    for i in order:
        r = pool.iloc[i]
        if r["split"] == "train" or not MIN_WORDS <= len(clean_urdu(r["transcription"]).split()) <= MAX_WORDS:
            continue
        name = r["path"] if "path" in pool.columns and r["path"] else r["audio"]["path"]
        clip_id = f"fleurs/{r['split']}/{os.path.basename(name).rsplit('.', 1)[0]}"
        audio, sr = sf.read(io.BytesIO(r["audio"]["bytes"]), dtype="float32")
        rows.append({"clip_id": clip_id, "dataset": "FLEURS", "subset": r["split"], "category": "",
                     "duration_s": save_audio(clip_id, to_16k(audio, sr)), "reference": r["transcription"]})
        if len(rows) == N_CANDIDATES:
            break
    return rows


clips = pd.DataFrame(fleurs_candidates())
assert clips.clip_id.is_unique, "clip_id is not unique"
clips["included"] = clips.duration_s <= MAX_CLIP_SECONDS
clips["excluded_reason"] = np.where(clips.included, "", f"longer than {MAX_CLIP_SECONDS} s")
os.makedirs(CANDIDATE_DIR, exist_ok=True)
clips.to_csv(SAMPLE_FILES["candidates"], index=False, encoding="utf-8")
print(clips.groupby("subset").agg(clips=("clip_id", "size"), median_s=("duration_s", "median"), longest_s=("duration_s", "max")).to_string())

## 3. Omnilingual and Whisper (Python 3.10 environment)

| Recogniser | Decoding | Word confidence |
|---|---|---|
| Omnilingual | `ASRInferencePipeline` defaults, one clip at a time | Not available: the pipeline computes beam scores internally but returns only text |
| Whisper | Model's default generation settings (greedy), language Urdu | Saved for clips up to 30 s: each word's probability is the product of its tokens' probabilities |

Whisper clips longer than 30 s use Whisper's long-form mode, which transcribes 30 s windows in sequence.
A Whisper output that stops at the 448-token limit instead of an end token is saved with status `cut_off`.

In [ ]:
%%writefile transcribe_310.py
import sys
import numpy as np, soundfile as sf, torch
from baseline_common import SAMPLE_RATE, run_recogniser


def read(path):
    return sf.read(path, dtype="float32")[0]


def omnilingual():
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
    pipe = ASRInferencePipeline(model_card="omniASR_LLM_1B_v2", device="cuda", dtype=torch.float16)
    beam = {k: repr(v) for k, v in vars(pipe).items() if "beam" in k.lower()}
    print("Omnilingual beam settings:", beam or "none exposed")
    decoding = f"pipeline defaults; {beam}" if beam else "pipeline defaults"

    def one(path, duration_s):
        text = pipe.transcribe([{"waveform": read(path), "sample_rate": SAMPLE_RATE}],
                               lang=["urd_Arab"], batch_size=1)[0]
        return {"text": text, "decoding": decoding, "word_confidence": None, "status": "ok"}

    run_recogniser("omnilingual_llm_1b_v2", one)


def whisper():
    from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
    name = "openai/whisper-large-v3"
    proc = AutoProcessor.from_pretrained(name)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(name, torch_dtype=torch.float16,
                                                      low_cpu_mem_usage=True).to("cuda").eval()
    tok = proc.tokenizer
    special = set(tok.all_special_ids)
    eos = model.generation_config.eos_token_id
    beams = model.generation_config.num_beams or 1
    print(f"Whisper generation: num_beams={beams}, max_length={model.generation_config.max_length}")

    def word_confidence(sequence, scores):
        # Returns [[word, probability], ...]. A new word starts at a token that begins with a space;
        # a word's probability is the product of its tokens' probabilities. Special tokens are skipped.
        generated = sequence[-len(scores):].tolist()
        words, ids, prob = [], [], 1.0
        for token, step in zip(generated, scores):
            if token in special:
                continue
            p = torch.softmax(step[0].float(), dim=-1)[token].item()
            if tok.convert_ids_to_tokens(token).startswith("Ġ") and ids:
                words.append([tok.decode(ids).strip(), round(prob, 4)])
                ids, prob = [], 1.0
            ids.append(token)
            prob *= p
        if ids:
            words.append([tok.decode(ids).strip(), round(prob, 4)])
        return words

    def one(path, duration_s):
        audio = read(path)
        if duration_s <= 30:
            feats = proc(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt").input_features.to("cuda", torch.float16)
            with torch.no_grad():
                out = model.generate(feats, language="urdu", task="transcribe",
                                     return_dict_in_generate=True, output_scores=True)
            seq = out.sequences[0]
            return {"text": tok.decode(seq, skip_special_tokens=True),
                    "decoding": f"greedy, num_beams={beams}",
                    "word_confidence": word_confidence(seq, out.scores),
                    "status": "ok" if seq[-1].item() == eos else "cut_off"}
        inputs = proc(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt", truncation=False,
                      padding="longest", return_attention_mask=True).to("cuda", torch.float16)
        with torch.no_grad():
            ids = model.generate(**inputs, language="urdu", task="transcribe", return_timestamps=True)
        return {"text": proc.batch_decode(ids, skip_special_tokens=True)[0],
                "decoding": f"long-form 30 s windows, num_beams={beams}",
                "word_confidence": None, "status": "ok"}

    run_recogniser("whisper_large_v3", one)


if __name__ == "__main__":
    {"omnilingual": omnilingual, "whisper": whisper}[sys.argv[1]]()

In [ ]:
sh(f"cd /kaggle/working && {PY} -u transcribe_310.py omnilingual")

In [ ]:
sh(f"cd /kaggle/working && {PY} -u transcribe_310.py whisper")

## 4. Indic Transcribe Core (default Python)

Language `ur`. Every clip is transcribed in one pass with `asr(path, lang="ur")`.

In [ ]:
import torch
from huggingface_hub import snapshot_download

model_dir = snapshot_download("bodhan-ai/indic-transcribe-core", ignore_patterns=["nemo/*", "*.png"])
sys.path.insert(0, model_dir)
from indic_transcribe import IndicTranscribe

asr = IndicTranscribe.from_pretrained(model_dir)
if hasattr(asr, "to"):
    asr = asr.to("cuda")

In [ ]:
def as_text(output):
    if isinstance(output, str):
        return output
    if isinstance(output, (list, tuple)):
        return as_text(output[0])
    if isinstance(output, dict):
        return output.get("text") or output.get("transcript") or str(output)
    return getattr(output, "text", str(output))


def indic_one(path, duration_s):
    return {"text": as_text(asr(path, lang="ur")), "decoding": "single pass, model defaults",
            "word_confidence": None, "status": "ok"}


run_recogniser("indic_transcribe_core", indic_one)
del asr
torch.cuda.empty_cache()

## 5. Run information

In [ ]:
import transformers

env310 = subprocess.run(f"{PY} -c \"import torch, transformers, omnilingual_asr as o; "
                        f"print(torch.__version__, transformers.__version__, getattr(o, '__version__', '0.2.0'))\"",
                        shell=True, capture_output=True, text=True).stdout.split()
gpu = subprocess.run("nvidia-smi --query-gpu=name,driver_version --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
info = {
    "machine": f"Kaggle, {gpu.splitlines()[0]}",
    "normaliser": NORMALISER_VERSION,
    "selection": {"seed": SEED, "skipped_first_clips": SKIP_FIRST, "candidates": N_CANDIDATES, "words": [MIN_WORDS, MAX_WORDS]},
    "environments": {
        "python_3_10 (Omnilingual, Whisper)": {"torch": env310[0], "transformers": env310[1], "omnilingual_asr": env310[2]},
        "kaggle_default (Indic Transcribe)": {"python": sys.version.split()[0], "torch": torch.__version__,
                                              "transformers": transformers.__version__},
    },
    "models": {"whisper_large_v3": "openai/whisper-large-v3", "omnilingual_llm_1b_v2": "omniASR_LLM_1B_v2",
               "indic_transcribe_core": "bodhan-ai/indic-transcribe-core"},
}
json.dump(info, open(f"{CANDIDATE_DIR}/run_info.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(json.dumps(info, indent=2, ensure_ascii=False))

shutil.make_archive("/kaggle/working/output", "zip", OUT)
print("\nSaved /kaggle/working/output.zip")